<a href="https://colab.research.google.com/github/wikiwa1/ASD-with-SSL/blob/main/Baseline_model_with_logmel_spectrogram_avg_and_aucroc_metric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── imports ──────────────────────────────────────────────────────────────────
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import Audio, display

# ── global style ─────────────────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 11

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT = Path('/content/drive/MyDrive/fan/')   # adjust if data lives elsewhere
MACHINE_IDS = ['id_00', 'id_02', 'id_04', 'id_06']
SR = 16_000               # native sample rate
CHANNEL = 0               # which mic channel to use (0–7); channel 0 is conventional

In [4]:
logmel_df = pd.read_pickle('/content/drive/MyDrive/fan/logmel_df.pkl')

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    logmel_df,
    test_size=0.2,
    stratify=logmel_df['label'],
    random_state=42
)

In [16]:
import numpy as np

def pad_to_max(arrays):
    max_time = max(a.shape[1] for a in arrays)
    padded = [
        np.pad(a, ((0,0),(0, max_time - a.shape[1])),
               mode='constant', constant_values=a.min())
        for a in arrays
    ]
    return np.stack(padded)

averages = {}

for label, group in train_df.groupby('label'):
    stacked = pad_to_max(group['logmel'].tolist())
    avg_logmel = stacked.mean(axis=0)
    averages[label] = avg_logmel
    print(label, avg_logmel.shape)

avg_normal = averages['normal']
avg_abnormal = averages['abnormal']

abnormal (128, 313)
normal (128, 313)


In [13]:
print(type(avg_logmel))

<class 'numpy.ndarray'>


In [17]:
from sklearn.metrics import roc_auc_score

true_labels = [1 if label == 'abnormal' else 0 for label in test_df['label']]
scores = [np.linalg.norm(sample - avg_normal) for sample in test_df['logmel']]

auc = roc_auc_score(true_labels, scores)
print(auc)

0.5270229801393366


In [18]:
from sklearn.metrics import roc_auc_score

true_labels = [1 if label == 'abnormal' else 0 for label in test_df['label']]
scores = [- np.linalg.norm(sample - avg_abnormal) for sample in test_df['logmel']]

auc = roc_auc_score(true_labels, scores)
print(auc)

0.489366746386607


In [19]:
from sklearn.metrics import roc_auc_score

true_labels = [1 if label == 'abnormal' else 0 for label in test_df['label']]
scores = [np.linalg.norm(sample - avg_normal) - np.linalg.norm(sample - avg_abnormal) for sample in test_df['logmel']]

auc = roc_auc_score(true_labels, scores)
print(auc)

0.55807008422585


Trying the cosine distance

In [20]:
from sklearn.metrics import roc_auc_score
from scipy.spatial.distance import cosine

true_labels = [1 if label == 'abnormal' else 0 for label in test_df['label']]
scores = [cosine(sample.flatten(), avg_normal.flatten()) for sample in test_df['logmel']]

auc = roc_auc_score(true_labels, scores)
print(auc)

0.4818924820630134
